In [1]:
from pyspark.sql import SparkSession
import os
import json
from pprint import pprint
warehouse_path = r"C:\iceberg-warehouse"

print(os.listdir(warehouse_path))

['db']


In [2]:
spark = SparkSession.builder \
    .appName("IcebergLocal") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    ) \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "file:///C:/iceberg-warehouse"
    ) \
    .getOrCreate()

In [7]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SELECT current_catalog()").show(truncate=False)
spark.sql("""
SHOW TABLES
""").show(truncate=False)


+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+

+-----------------+
|current_catalog()|
+-----------------+
|spark_catalog    |
+-----------------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [9]:
spark.conf.get("spark.sql.extensions")

'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'

In [10]:
for k, v in spark.sparkContext.getConf().getAll():
    if "spark.sql.catalog" in k:
        print(k, "=", v)

spark.sql.catalog.local.type = hadoop
spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.local.warehouse = file:///C:/iceberg-warehouse


In [11]:
spark.sql("""
CREATE NAMESPACE IF NOT EXISTS local.demo
""")

DataFrame[]

In [12]:
spark.sql("""
CREATE TABLE local.demo.members (
    id BIGINT,
    registration_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (years(registration_ts))
""")

DataFrame[]

In [14]:
spark.sql("""
INSERT INTO local.demo.members VALUES
(1, TIMESTAMP '2023-01-15 10:00:00'),
(2, TIMESTAMP '2024-06-20 12:00:00')
""")

DataFrame[]

In [19]:
spark.sql("""
SELECT *
FROM local.demo.members.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+---------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                        |summary                                                                                                                                                                                                                                                                                         |
+-----------------------+-------------------+---------

In [16]:
spark.sql("""
SELECT *
FROM local.demo.members.files
""").show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------+-----------+-------+---------+------------+------------------+------------------+----------------+-----------------+----------------+----------------------------------------------------------------+----------------------------------------------------------------+------------+-------------+------------+-------------+------------------------------------------------------------------------------------+
|content|file_path                                                                                                                          |file_format|spec_id|partition|record_count|file_size_in_bytes|column_sizes      |value_counts    |null_value_counts|nan_value_counts|lower_bounds                                                    |upper_bounds                                                    |key_metadata|split_offsets|equality_ids|sort_order_i

In [17]:
spark.sql("""
ALTER TABLE local.demo.members
ADD PARTITION FIELD months(registration_ts)
""")

DataFrame[]

In [20]:
spark.sql("""
SELECT *
FROM local.demo.members.files
""").show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------+-----------+-------+----------+------------+------------------+------------------+----------------+-----------------+----------------+----------------------------------------------------------------+----------------------------------------------------------------+------------+-------------+------------+-------------+------------------------------------------------------------------------------------+
|content|file_path                                                                                                                          |file_format|spec_id|partition |record_count|file_size_in_bytes|column_sizes      |value_counts    |null_value_counts|nan_value_counts|lower_bounds                                                    |upper_bounds                                                    |key_metadata|split_offsets|equality_ids|sort_order

In [21]:
spark.sql("""
INSERT INTO local.demo.members VALUES
(3, TIMESTAMP '2026-03-10 08:00:00'),
(4, TIMESTAMP '2026-04-12 09:00:00')
""")

DataFrame[]

In [22]:
spark.sql("""
SELECT
    spec_id,
    file_path,
    partition
FROM local.demo.members.files
""").show(truncate=False)

+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|spec_id|file_path                                                                                                                                                         |partition |
+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|1      |file:/C:/iceberg-warehouse/demo/members/data/registration_ts_year=2026/registration_ts_month=2026-04/00000-13-17956af3-bc3c-46a7-86db-a89c9f9e6d23-0-00002.parquet|{56, 675} |
|1      |file:/C:/iceberg-warehouse/demo/members/data/registration_ts_year=2026/registration_ts_month=2026-03/00000-13-17956af3-bc3c-46a7-86db-a89c9f9e6d23-0-00001.parquet|{56, 674} |
|0      |file:/C:/iceberg-warehouse/demo/members/data/registration_ts_year=2024/